In [1]:
!pip install emoji

In [2]:
import os
import sys

# 1. 자바 환경 변수 강제 설정 (사용자님의 실제 경로 반영)
# jdk17.0.17_10 폴더 안의 bin\server 폴더에 jvm.dll이 있습니다.
java_path = r'C:\Program Files\Amazon Corretto\jdk17.0.17_10'
os.environ['JAVA_HOME'] = java_path
os.environ['PATH'] = os.path.join(java_path, 'bin') + os.path.pathsep + os.environ['PATH']

# 2. 스트림릿 라이브러리 임포트 (NameError 방지)
import streamlit as st

# 3. AI 엔진 로드 시도
try:
    from LLMResponse import LLMResponse
    bot = LLMResponse()
    llm_ready = True
    print("✅ AI 엔진 로드 성공!")
except Exception as e:
    print(f"❌ 오류 발생: {e}")
    llm_ready = False

items : 83, colours : 24, materials : 23
✅ AI 엔진 로드 성공!


In [3]:
%%writefile lookxpert_final.py
import streamlit as st
import os
import sys
import time
import urllib.parse

# -------------------------------------------------
# ⚙️ 1. 시스템 설정
# -------------------------------------------------
st.set_page_config(page_title="LookXpertM", layout="wide")

java_path = r'C:\Program Files\Amazon Corretto\jdk17.0.17_10'
os.environ['JAVA_HOME'] = java_path
os.environ['PATH'] = os.path.join(java_path, 'bin') + os.path.pathsep + os.environ['PATH']

# -------------------------------------------------
# 🔍 2. AI 엔진 로드 (성능 최우선 모드)
# -------------------------------------------------
@st.cache_resource
def load_bot():
    try:
        from LLMResponse import LLMResponse
        return LLMResponse()
    except Exception as e:
        return None

bot = load_bot()

# -------------------------------------------------
# 👗 3. 메인 화면 구성
# -------------------------------------------------
st.title("👗 LookXpertM – 패션 추천 전문가")
st.caption("상태: ✅ 시스템 최적화 완료")

tab1, tab2, tab3 = st.tabs(["✨ 스마트 추천", "🛍️ 무신사 검색", "🔗 트렌드 링크"])

with tab1:
    c1, c2, c3 = st.columns(3)
    w = c1.selectbox("날씨", ["보통", "추움", "더움"])
    s = c2.selectbox("상황", ["데이트", "출근", "캐주얼"])
    t = c3.selectbox("톤", ["웜톤", "쿨톤"])
    
    # 💡 팁: 버튼을 누르면 '즉시' 응답을 받기 위해 max_tokens를 50으로 극단적으로 낮춤
    if st.button("🎯 추천받으세요", use_container_width=True):
        if bot:
            with st.spinner("최적의 추천 조합을 실시간 생성 중..."):
                # 아주 짧고 명확한 답변만 요구하여 속도를 높임
                res = bot.GetLLMResponse(f"{w} 날씨 {s}에 어울리는 {t} 코디 3가지 핵심만 제안해줘.", max_tokens=50)
                st.success("분석 완료!")
                st.info(res)
        else:
            st.error("엔진 로드에 실패했습니다. 자바 경로를 확인하세요.")

with tab2:
    search_kw = st.text_input("찾고 싶은 아이템", "겨울 롱코트")
    if search_kw:
        url = f"https://www.musinsa.com/search/goods?keyword={urllib.parse.quote(search_kw)}"
        st.link_button(f"🚀 무신사에서 '{search_kw}' 결과 확인", url, use_container_width=True)

with tab3:
    col1, col2 = st.columns(2)
    col1.link_button("📰 VOGUE 트렌드", "https://www.vogue.co.kr/category/fashion/fashion-trend/")
    col2.link_button("📰 ELLE 패션", "https://www.elle.co.kr/Fashion")

Overwriting lookxpert_final.py


In [ ]:
%%writefile LLMResponse.py
from FashionTrendCrawling import FashionTrendCrawling as ftc
from KeywordCounter import KeywordCounter as kc
from FashionChatbot import FashionChatbot
from IPython.display import clear_output
import pandas as pd
import time
import os

class LLMResponse:
    def __init__(self):
        self.__counter = kc(
            items = ["코트", "패딩", "자켓", "점퍼", "블레이저", "가디건", "니트", "스웨터", "바지", "청바지", "슬랙스", "운동화", "머플러"],
            colours = ["블랙", "화이트", "아이보리", "베이지", "그레이", "네이비", "브라운", "카키"],
            materials = ["울", "캐시미어", "가죽", "코튼", "면", "데님", "나일론"]
        )
        self.__bot = None

    def GetLLMResponse(self, userInput:str, model_name:str = "openai:gpt-4.1-nano", max_tokens:int = 512) -> str:
        self.__counter.ClearCounts()
        
        # 파일명 규칙: 스트림릿 앱의 query와 일치해야 함
        safe_query = userInput.replace(" ", "_").replace("/", "_")
        file_path = f"data_{safe_query}.csv"
        
        if os.path.exists(file_path):
            data = pd.read_csv(file_path)
            for text in data['text'].dropna():
                self.__counter.BeginCounting(text)
        else:
            # 파일이 없으면 크롤링 수행
            data = self.__NaverBlogCrawiling(userInput.replace(" ", "+"))
            data.to_csv(file_path, index=False, encoding='utf-8-sig')
        
        counts = self.__counter.GetCounts()
        if self.__bot == None:
            self.__bot = FashionChatbot(model_name = model_name, max_tokens = max_tokens)
        
        clear_output(wait = True)
        
        # 챗봇 응답 요청
        response = self.__bot.RequestResponse(
            f"추천 키워드 기반 코디 제안: {userInput}"
        )
        return f"결과 >> {response.content}"

    def __NaverBlogCrawiling(self, query:str) -> pd.DataFrame:
        # 이 부분이 실패하면 빈 데이터라도 반환하게 예외 처리
        try:
            elements = ftc.BeginCrawling(
                url = f"https://search.naver.com/search.naver?query={query}",
                selectors = ["div.api_txt_lines"],
                element_names = ["text"],
                timeout = 1000,
                scrolling = False
            )
            return pd.DataFrame(elements)
        except:
            return pd.DataFrame({'link': ['#'], 'text': ['데이터 없음']})

In [ ]:
!streamlit run lookxpert_final.py